In [84]:
from fredapi import Fred
import pandas as pd
import numpy as np
import yfinance as yf
from pandas_datareader.data import DataReader


API_KEY = '60c97d27ab6c225f88849013d178b0d9'


# Grabbing data with FRED API
Here I download 13 raw datapoints which we can use for the JDKF

I didn't know what these were, writing down for me: 
- Core CPI is CPI minus food and fuel prices, because these can move around a lot due to wars etc.
- PPI is the producer price index which is what producers are paying for their supplies
- BAA yield is the yield of corperate bonds from companies rated 'Baa' by Moody's

In [ ]:
fred = Fred(api_key=API_KEY)

series_codes = {
    "real_gdp": "GDPC1",
    "industrial_production": "INDPRO",
    "unemployment_rate": "UNRATE",
    "nonfarm_payrolls": "PAYEMS",
    "cpi": "CPIAUCSL",
    "core_cpi": "CPILFESL",
    "ppi": "PPIACO",
    "fed_funds_rate": "FEDFUNDS",
    "baa_yield": "BAA",
    "ten_year_yield": "GS10",
    "one_year_yield": "GS1",
    "house_price_index": "QUSR628BIS",
    "oil_price": "WTISPLC",
}

df = pd.DataFrame({
    name: fred.get_series(code)
    for name, code in series_codes.items()
})


df = df.loc["1970-01-01":"2025-12-31"]

I am deciding wheter we want to create quarterly data by averaging or just taking the last value of the quarter. I think that if we want to capture shocks then having the last datapoint of the quarter would be better. I think both sides can be justified for a lot of these factors, maybe we take an average and a current?

In [ ]:
mean_cols = [
    "real_gdp",
    "industrial_production",
    "unemployment_rate",
    "nonfarm_payrolls",
    "cpi",
    "core_cpi",
    "ppi",
    "fed_funds_rate",
    "baa_yield",
    "ten_year_yield",
    "one_year_yield"

]

last_cols = [
    "house_price_index",
    "oil_price"
]

quarterly_mean = df[mean_cols].resample("QE").mean()
quarterly_last = df[last_cols].resample("QE").last()

quarterly = pd.concat(
    [quarterly_mean, quarterly_last],
    axis=1
)

FRED only had daily sp500 data going back to 2016, I use yfinance to get sp500 data going back to 1970

In [ ]:
sp_raw = yf.download(
    "^GSPC",
    start="1970-01-01",
    auto_adjust=True,
    progress=False
)

sp = sp_raw["Close"]

# Convert from one-column DataFrame to Series if needed
if isinstance(sp, pd.DataFrame):
    sp = sp.squeeze()

sp.name = "sp500"
sp.index = pd.to_datetime(sp.index)

sp_q = sp.resample("QE").last()

quarterly = quarterly.join(sp_q, how="left")

In [117]:
quarterly.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 224 entries, 1970-03-31 to 2025-12-31
Freq: QE-DEC
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   real_gdp               224 non-null    float64
 1   industrial_production  224 non-null    float64
 2   unemployment_rate      224 non-null    float64
 3   nonfarm_payrolls       224 non-null    float64
 4   cpi                    224 non-null    float64
 5   core_cpi               224 non-null    float64
 6   ppi                    224 non-null    float64
 7   fed_funds_rate         224 non-null    float64
 8   baa_yield              224 non-null    float64
 9   ten_year_yield         224 non-null    float64
 10  one_year_yield         224 non-null    float64
 11  oil_price              224 non-null    float64
 12  house_price_index      224 non-null    float64
 13  sp500                  224 non-null    float64
dtypes: float64(14)
memory usag

### Feature engineering